In [ ]:
# Start your code here!
import os
import json
import pandas as pd
from openai import OpenAI

# Instantiate an API client
client = OpenAI()

In [ ]:
# Step 1: Build nasdaq100_ca and add the ytd column
nasdaq100_ca = pd.read_csv("nasdaq100_CA.csv")
ytd = pd.read_csv("nasdaq100_price_change.csv")[["symbol", "ytd"]]
nasdaq100_ca = nasdaq100_ca.merge(ytd, on="symbol")

In [ ]:
# Step 2: Classify all stocks into sectors with a SINGLE OpenAI API call
SECTORS = "Technology, Consumer Cyclical, Industrials, Utilities, Healthcare, Communication, Energy, Consumer Defensive, Real Estate, Financial"
companies = nasdaq100_ca[["symbol", "name"]].to_dict(orient="records")

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": (
        f"Classify each company below into one of these sectors: {SECTORS}.\n"
        f"Return ONLY a JSON object mapping symbol to sector.\n\n{json.dumps(companies)}"
    )}],
    temperature=0,
    response_format={"type": "json_object"},
)

sectors = json.loads(response.choices[0].message.content)
nasdaq100_ca["sector"] = nasdaq100_ca["symbol"].map(sectors)
nasdaq100_ca

In [ ]:
# Step 3: Use the OpenAI API to recommend the two best sectors and companies
data = nasdaq100_ca[["symbol", "name", "sector", "ytd"]].to_string(index=False)

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": (
        "Based on the YTD performance of these California-headquartered Nasdaq-100 companies, "
        "recommend the TWO best-performing sectors and at least TWO companies per sector with a brief rationale.\n\n"
        f"{data}"
    )}],
    temperature=0.3,
)

stock_recommendations = response.choices[0].message.content
print(stock_recommendations)